# Data Loading, Cleaning
## Austin Animal Center · Outcomes + Intakes

**Goal:** Load both datasets from the Socrata API, fix all known data quality issues,
engineer baseline features, and explore the distributions that shape Week 2 modeling decisions.

**Binary target:** Adopted (1) vs Not Adopted (0)  
**Multi-class target:** Adopted / Transferred / Reclaimed / Euthanized / Died / Other

## Setup

Run the install cell once if any packages are missing, then **restart the kernel**.
Packages already installed (pandas, numpy) are safe to include — pip is idempotent.

In [80]:
# Run this cell once if needed, then restart the kernel before continuing.
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn requests

In [81]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print(f'pandas  {pd.__version__}')
print(f'numpy   {np.__version__}')

pandas  3.0.0
numpy   2.4.1


## Data Loading

Local copies of the full historical exports (2013-10-01 to 2025-05-05) downloaded
from the Austin data portal UI.

In [82]:
FULL_OUTCOMES_PATH = Path('../data/raw/Austin_Animal_Center_Outcomes_(10_01_2013_to_05_05_2025)_20260523.csv')
FULL_INTAKES_PATH  = Path('../data/raw/Austin_Animal_Center_Intakes_(10_01_2013_to_05_05_2025)_20260523.csv')

df_full_outcomes = pd.read_csv(FULL_OUTCOMES_PATH)
df_full_intakes  = pd.read_csv(FULL_INTAKES_PATH)

print('=== FULL OUTCOMES EXPORT ===')
df_full_outcomes.info()
print('\n--- head(3) ---')
display(df_full_outcomes.head(3))

print('\n=== FULL INTAKES EXPORT ===')
df_full_intakes.info()
print('\n--- head(3) ---')
df_full_intakes.head(3)

=== FULL OUTCOMES EXPORT ===
<class 'pandas.DataFrame'>
RangeIndex: 173775 entries, 0 to 173774
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Animal ID         173775 non-null  str  
 1   Date of Birth     173775 non-null  str  
 2   Name              123991 non-null  str  
 3   DateTime          173775 non-null  str  
 4   MonthYear         173775 non-null  str  
 5   Outcome Type      173729 non-null  str  
 6   Outcome Subtype   79660 non-null   str  
 7   Animal Type       173775 non-null  str  
 8   Sex upon Outcome  173774 non-null  str  
 9   Age upon Outcome  173766 non-null  str  
 10  Breed             173775 non-null  str  
 11  Color             173775 non-null  str  
dtypes: str(12)
memory usage: 15.9 MB

--- head(3) ---


,Animal ID,Date of Birth,Name,DateTime,MonthYear,Outcome Type,Outcome Subtype,Animal Type,Sex upon Outcome,Age upon Outcome,Breed,Color
0,A668305,2012-12-01,NaN,2013-12-02T00:00:00-05:00,12-2013,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Brown/Yellow
1,A673335,2012-02-22,NaN,2014-02-22T00:00:00-05:00,02-2014,Euthanasia,Suffering,Other,Unknown,2 years,Raccoon,Black/Gray
2,A675999,2013-04-03,NaN,2014-04-07T00:00:00-05:00,04-2014,Transfer,Partner,Other,Unknown,1 year,Turtle Mix,Green



=== FULL INTAKES EXPORT ===
<class 'pandas.DataFrame'>
RangeIndex: 173812 entries, 0 to 173811
Data columns (total 12 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   Animal ID         173812 non-null  str  
 1   Name              123821 non-null  str  
 2   DateTime          173812 non-null  str  
 3   MonthYear         173812 non-null  str  
 4   Found Location    173812 non-null  str  
 5   Intake Type       173812 non-null  str  
 6   Intake Condition  173812 non-null  str  
 7   Animal Type       173812 non-null  str  
 8   Sex upon Intake   173811 non-null  str  
 9   Age upon Intake   173812 non-null  str  
 10  Breed             173812 non-null  str  
 11  Color             173812 non-null  str  
dtypes: str(12)
memory usage: 15.9 MB

--- head(3) ---


,Animal ID,Name,DateTime,MonthYear,Found Location,Intake Type,Intake Condition,Animal Type,Sex upon Intake,Age upon Intake,Breed,Color
0,A521520,Nina,10/01/2013 07:51:00 AM,October 2013,Norht Ec in Austin (TX),Stray,Normal,Dog,Spayed Female,7 years,Border Terrier/Border Collie,White/Tan
1,A664235,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White
2,A664236,NaN,10/01/2013 08:33:00 AM,October 2013,Abia in Austin (TX),Stray,Normal,Cat,Unknown,1 week,Domestic Shorthair Mix,Orange/White


## Clean & Merge Full-Export CSVs into One Frame

Align the 173k historical CSV rows to a snake_case schema, then backward-asof
merge intakes ← outcomes so each outcome carries its triggering intake's
context. Output: `df_full_merged` — one row per outcome event.

### Step 1: change the column names into sanke_schema for further data cleaning

In [83]:
# --- outcomes side: rename to target schema ---
o = df_full_outcomes.rename(columns={
    'Animal ID':       'animal_id',
    'Name':            'name',
    'DateTime':        'outcome_date',
    'Date of Birth':   'date_of_birth',
    'Outcome Type':    'outcome_type',
    'Outcome Subtype': 'outcome_subtype',
    'Animal Type':     'animal_type',
    'Color':           'color_raw',       # to be split in cell 4
    'Sex upon Outcome':'sex_upon_outcome',# to be split in cell 4
})

o = o.drop(columns=['MonthYear', 'Age upon Outcome', 'Breed'])

# --- intakes side: rename to target schema ---
i = df_full_intakes.rename(columns={
    'Animal ID':        'animal_id',
    'DateTime':         'intake_date',
    'Found Location':   'found_location',
    'Intake Type':      'intake_reason',
    'Intake Condition': 'intake_health_condition',
    'Breed':            'breed',
    'Sex upon Intake':  'sex_upon_intake', # to be processed in cell 4
})

# 'Name' kept from the outcomes side only, so drop it here to avoid a duplicate column
i = i.drop(columns=['Animal Type', 'Color', 'MonthYear', 'Age upon Intake', 'Name'])

print(f'After rename — outcomes: {o.shape}, intakes: {i.shape}')
print('Outcomes cols:', list(o.columns))
print('Intakes  cols:', list(i.columns))

After rename — outcomes: (173775, 9), intakes: (173812, 7)
Outcomes cols: ['animal_id', 'date_of_birth', 'name', 'outcome_date', 'outcome_type', 'outcome_subtype', 'animal_type', 'sex_upon_outcome', 'color_raw']
Intakes  cols: ['animal_id', 'intake_date', 'found_location', 'intake_reason', 'intake_health_condition', 'sex_upon_intake', 'breed']


### Step 2: Normalize & Validate dates
Convert mixed ISO datetimes to date-only (YYYY-MM-DD), stripping timezone offsets. Set any date_of_birth that is after outcome_date into null.

In [84]:
# Parse mixed ISO datetimes into date-only (YYYY-MM-DD)
# Reasons:
# - Source strings contain both timezone-aware (e.g. 2013-12-02T00:00:00-05:00) and naive ISO strings (e.g. 2013-10-01T09:31:00).
# - If pandas parses a Series as tz-aware, the naive strings can become NaT.
# - To avoid that, strip the timezone offset suffix, parse, then normalize to date.

def _parse_date_only(series: pd.Series) -> pd.Series:
    """Return a date-only Series (datetime64[ns]) from mixed ISO datetime strings."""
    series = series.astype(str).str.strip()
    # Remove timezone offsets like -05:00 or +00:00 so both offset and naive strings parse
    series = series.str.replace(r'([+-]\d{2}):(\d{2})$', '', regex=True)
    # Parse; errors='coerce' turns unparseable values into NaT, .dt.normalize() keeps date only
    return pd.to_datetime(series, errors='coerce').dt.normalize()

# Apply parsing to columns (works whether the column currently holds raw strings
# or Timestamp objects represented as strings)
o['outcome_date'] = _parse_date_only(o['outcome_date'])
o['date_of_birth'] = _parse_date_only(o['date_of_birth'])
i['intake_date'] = _parse_date_only(i['intake_date'])

# DOB after intake is impossible — null these (compare dates only)
mask = o['date_of_birth'] > o['outcome_date']
o.loc[mask, 'date_of_birth'] = pd.NaT

# Audits
print(f'Nulled impossible DOBs: {o["date_of_birth"].isna().sum():,}')
print(f'Null outcome dates: {o["outcome_date"].isna().sum():,}')
print(f'Null intake dates: {i["intake_date"].isna().sum():,}')

Nulled impossible DOBs: 33
Null outcome dates: 0
Null intake dates: 0


### Step 3: Parse sex, spay/neuter, color and breed fields

Extract `sex` from `sex_upon_outcome` — but **drop the outcome spay/neuter flag**: it's
target leakage (shelters spay/neuter as part of adoption, so the flag is caused by the
outcome). Split `color` and `breed` into primary/secondary colors and breeds, derive the
non-leaky boolean `is_previously_spayed_neutered` from `sex_upon_intake` (known at intake
time), and drop the raw source columns.

In [ ]:
SEX_UPON_MAP = {
    'Neutered Male': ('Male',    'Yes'),
    'Spayed Female': ('Female',  'Yes'),
    'Intact Male':   ('Male',    'No'),
    'Intact Female': ('Female',  'No'),
    'Unknown':       ('Unknown', 'Unknown'),
}


def _split_sex(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    mapped = series.map(SEX_UPON_MAP)
    sex  = mapped.map(lambda t: t[0] if isinstance(t, tuple) else 'Unknown')
    spay = mapped.map(lambda t: t[1] if isinstance(t, tuple) else 'Unknown')
    return sex, spay


def _split_slash(series: pd.Series) -> tuple[pd.Series, pd.Series]:
    parts = series.fillna('').str.split('/', n=1, expand=True)
    primary = parts[0].replace('', np.nan)
    # If no value contains '/', split yields a single column. Return an all-NaN
    # Series (not a bare scalar) so the secondary_color assignment stays aligned.
    if parts.shape[1] > 1:
        secondary = parts[1].replace('', np.nan)
    else:
        secondary = pd.Series(np.nan, index=series.index)
    return primary, secondary


def parse_breed(b):
    if pd.isna(b):
        return (np.nan, pd.NA)
    b = str(b).strip()
    # 1 = mixed/crossbreed (has " Mix" or a "/"), 0 = single breed
    is_mix = 1 if (('mix' in b.lower()) or ('/' in b)) else 0
    # Take first breed token (before "/" or " Mix")
    primary = b.split('/')[0].replace(' Mix', '').replace(' mix', '').strip()
    return (primary, is_mix)



# --- outcomes: split Sex upon Outcome -> sex only ---
o['sex'], _ = _split_sex(o['sex_upon_outcome'])
o['primary_color'], o['secondary_color'] = _split_slash(o['color_raw'])
o = o.drop(columns=['sex_upon_outcome', 'color_raw'])

# --- intakes: derive is_previously_spayed_neutered (1/0) from Sex upon Intake ---
_, spay_intake = _split_sex(i['sex_upon_intake'])
i['is_previously_spayed_neutered'] = spay_intake.map({'Yes': 1, 'No': 0}).astype('Int64')
i = i.drop(columns=['sex_upon_intake'])

# --- intakes: breed -> primary breed + is_mix (1/0) ---
parsed = i['breed'].apply(parse_breed)
i['primary_breed'] = [p[0] for p in parsed]
i['is_mix']        = pd.array([p[1] for p in parsed], dtype='Int64')
i = i.drop(columns=['breed'])


### Step 4: Merge intakes ← outcomes & engineer baseline features and srop no intake records

Backward `merge_asof` on `animal_id`, matching each outcome to the most recent
prior intake (`direction='backward'`), so every outcome row carries its
triggering intake's context. Output: `df_full_merged` — one row per outcome event.

In [86]:
intake_cols = [
    'animal_id', 'intake_date',
    'intake_reason', 'intake_health_condition', 'found_location',
    'is_previously_spayed_neutered', 'primary_breed', 'is_mix'
]

intakes_for_merge = (
    i[intake_cols]
    .sort_values('intake_date')
)
outcomes_sorted = o.sort_values('outcome_date')

df_full_merged = pd.merge_asof(
    outcomes_sorted,
    intakes_for_merge,
    by='animal_id',
    left_on='outcome_date',
    right_on='intake_date',
    direction='backward',
)


### Step 5: Drop unmatched outcomes and deduplicate merged data

Filter out merged outcome rows that have no matched prior `intake_date`, then remove duplicate rows from `df_full_merged`. Print the number of rows dropped and audit the final row count and missing intake dates.

In [87]:
# drop orphans with missing intake date (~0.5%)
before = len(df_full_merged)
df_full_merged = df_full_merged[df_full_merged['intake_date'].notna()].copy()
dropped = before - len(df_full_merged)
print(f'Dropped {dropped:,} rows missing intake_date '
      f'({dropped / before * 100:.2f}%) — {len(df_full_merged):,} rows remain')


# drop duplicates]
df_full_merged.drop_duplicates(inplace=True)

# Audits
print(f'Row parity (merged vs outcomes): {len(df_full_merged):,} vs {len(o):,}')
print(f'Missing intake_date:             {df_full_merged["intake_date"].isna().sum():,}')

Dropped 925 rows missing intake_date (0.53%) — 172,850 rows remain
Row parity (merged vs outcomes): 172,803 vs 173,775
Missing intake_date:             0


### Step 5: Group outcome_type into a multi-class target

Collapse the 12 raw `outcome_type` values into 5 modeling categories in a new
`outcome_category` column:

- **Adopted** — `Adoption`, `Rto-Adopt` (return-to-owner that converted to adoption)
- **Return to Owner** — `Return to Owner`
- **Transfer** — `Transfer`
- **Euthanasia** — `Euthanasia`
- **Other** — everything else (`Died`, `Disposal`, `Missing`, `Relocate`, `Stolen`, `Lost`, missing)

### Step 6: Compute baseline features
Add baseline features
- **`length_of_stay_days`** — `outcome_date − intake_date`
- **`age_at_intake_days`** — `intake_date − date_of_birth`
- **`is_adopted`** — binary target, 1 if `outcome_category == 'Adopted' else 0
- **`is_RTO`** — binary target, 1 if `outcome_category == 'Return to owner' else 0
- **`is_transfer`** — binary target, 1 if `outcome_category == 'Transfer' else 0
- **`is_euthanasia`** — binary target, 1 if `outcome_category == 'euthanasia' else 0

In [ ]:
# Derived columns
df_full_merged['length_of_stay_days'] = (
    df_full_merged['outcome_date'] - df_full_merged['intake_date']
).dt.days


df_full_merged['age_at_intake_days'] = (
    df_full_merged['intake_date'] - df_full_merged['date_of_birth']
).dt.days
before = len(df_full_merged)
df_full_merged = df_full_merged[~(df_full_merged['age_at_intake_days'] < 0)].copy()
dropped = before - len(df_full_merged)
print(f'Dropped {dropped:,} rows with negative age_at_intake_days '
      f'({dropped / before * 100:.2f}%) — {len(df_full_merged):,} rows remain')


# Group the 12 raw outcome_type values into 5 modeling categories.

OUTCOME_CATEGORY_MAP = {
    'Adoption':        'Adopted',
    'Rto-Adopt':       'Adopted',
    'Return to Owner': 'Return to Owner',
    'Transfer':        'Transfer',
    'Euthanasia':      'Euthanasia',
}

_category = df_full_merged['outcome_type'].map(OUTCOME_CATEGORY_MAP).fillna('Other')
CATEGORY_FLAGS = {
    'Adopted':         'is_adopted',
    'Return to Owner': 'is_RTO',
    'Transfer':        'is_transfer',
    'Euthanasia':      'is_euthanasia',
}
for category, col in CATEGORY_FLAGS.items():
    df_full_merged[col] = (_category == category).astype(int)

print(df_full_merged[list(CATEGORY_FLAGS.values())].sum())

df_full_merged['intake_year']      = df_full_merged['intake_date'].dt.year
df_full_merged['intake_month']     = df_full_merged['intake_date'].dt.month
df_full_merged['intake_dayofweek'] = df_full_merged['intake_date'].dt.dayofweek      # 0 = Monday

# Drop orphan outcomes with no matching prior intake (~0.5%).
before = len(df_full_merged)
df_full_merged = df_full_merged[df_full_merged['intake_date'].notna()].copy()
dropped = before - len(df_full_merged)
print(f'Dropped {dropped:,} rows missing intake_date '
      f'({dropped / before * 100:.2f}%) — {len(df_full_merged):,} rows remain')


# Audits on the engineered features
print(f'Negative LOS: {(df_full_merged["length_of_stay_days"] < 0).sum():,}')
print(f'Median LOS:   {df_full_merged["length_of_stay_days"].median():.0f} days')


Dropped 191 rows with negative age_at_intake_days (0.11%) — 172,612 rows remain
is_adopted       85094
is_RTO           25635
is_transfer      48392
is_euthanasia    10797
dtype: int64
Dropped 0 rows missing intake_date (0.00%) — 172,612 rows remain
Negative LOS: 0
Median LOS:   6 days


### Step 7: Store processed data into csv

In [91]:
# Reorder columns to the requested canonical order
desired_order = [
    'animal_id',
    'name',
    'animal_type',
    'sex',
    'primary_breed',
    'is_mix',
    'date_of_birth',
    'primary_color',
    'secondary_color',
    'intake_date',
    'intake_reason',
    'found_location',
    'age_at_intake_days',
    'intake_health_condition',
    'is_previously_spayed_neutered',
    'outcome_date',
    'outcome_type',
    'outcome_subtype',
    'length_of_stay_days',
    'is_adopted',
    'is_RTO',
    'is_transfer',
    'is_euthanasia',
    'intake_year',
    'intake_month',
    'intake_dayofweek'
]

existing = [c for c in desired_order if c in df_full_merged.columns]
missing = [c for c in desired_order if c not in df_full_merged.columns]
if missing:
    print('Warning: missing columns not found and will be skipped:', missing)

# Preserve any other columns that are not in the requested list
other_cols = [c for c in df_full_merged.columns if c not in existing]

# Reassign with new column order
df_full_merged = df_full_merged[existing + other_cols]
print('Reordered df_full_merged columns — first columns now:')
print(df_full_merged.columns.tolist()[:len(existing)])

processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

out_path = processed_dir / 'df_full_merged.csv'
df_full_merged.to_csv(out_path, index=False)

print(f'Saved {len(df_full_merged):,} rows to {out_path}')

Reordered df_full_merged columns — first columns now:
['animal_id', 'name', 'animal_type', 'sex', 'primary_breed', 'is_mix', 'date_of_birth', 'primary_color', 'secondary_color', 'intake_date', 'intake_reason', 'found_location', 'age_at_intake_days', 'intake_health_condition', 'is_previously_spayed_neutered', 'outcome_date', 'outcome_type', 'outcome_subtype', 'length_of_stay_days', 'is_adopted', 'is_RTO', 'is_transfer', 'is_euthanasia', 'intake_year', 'intake_month', 'intake_dayofweek']
Saved 172,612 rows to ../data/processed/df_full_merged.csv


In [92]:
#check unique values of animal_type and filter to contain only dogs and cats for EDA
df_full_merged['animal_type'].unique() 
df_full_small_animal= df_full_merged[df_full_merged['animal_type'].isin(['Dog', 'Cat'])]

In [93]:
df_full_small_animal.info()

<class 'pandas.DataFrame'>
Index: 162765 entries, 1 to 173774
Data columns (total 27 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   animal_id                      162765 non-null  str           
 1   name                           121169 non-null  str           
 2   animal_type                    162765 non-null  str           
 3   sex                            162765 non-null  str           
 4   primary_breed                  162765 non-null  str           
 5   is_mix                         162765 non-null  Int64         
 6   date_of_birth                  162734 non-null  datetime64[us]
 7   primary_color                  162765 non-null  str           
 8   secondary_color                87231 non-null   str           
 9   intake_date                    162765 non-null  datetime64[us]
 10  intake_reason                  162765 non-null  str           
 11  found_location  